# Mini-Projet NLP : Transformers et Système RAG

Ce notebook présente une exploration pratique des modèles **Transformers** pour différentes tâches de traitement automatique du langage naturel, puis l’implémentation d’un système **RAG** simple permettant d’améliorer les réponses d’un modèle de langage à l’aide d’un corpus externe.

Le projet est divisé en deux grandes parties :

1. Exploration des Transformers et tâches NLP.
2. Systèmes RAG (Retrieval-Augmented Generation).

## Environnement et bibliothèques utilisées

Dans ce notebook, nous utiliserons principalement les bibliothèques suivantes :

* `transformers` : pour charger et utiliser des modèles pré-entraînés.
* `sentence-transformers` : pour générer les embeddings des documents et des requêtes.
* `faiss` : pour indexer les embeddings et effectuer une recherche de similarité.
* `torch` : pour l’exécution des modèles de deep learning.
* `pandas` et `numpy` : pour la manipulation des données.
* `PyPDF2` ou `pypdf` : pour lire le contenu des documents PDF si nécessaire.

Ces bibliothèques permettent de construire une chaîne complète allant du traitement du texte jusqu’à la génération de réponses augmentées par des documents externes.

## Partie 1 : Exploration des Transformers et tâches NLP

Les Transformers sont des architectures de deep learning très utilisées en NLP.  
Ils permettent de traiter du texte en capturant les relations entre les mots grâce au mécanisme d’attention.

Dans cette partie, nous allons utiliser des modèles pré-entraînés disponibles sur HuggingFace afin de réaliser plusieurs tâches NLP sans entraîner un modèle depuis zéro.

In [1]:
# Installation des bibliothèques nécessaires
!pip install datasets accelerate
!pip install -q transformers==4.44.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 134.8 MB/s eta 0:00:00


## 1-a Importation des bibliothèques

In [2]:
import torch
from transformers import AutoTokenizer, AutoModel, pipeline

device = 0 if torch.cuda.is_available() else -1

print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

GPU disponible : True
Nom du GPU : Tesla T4


## 1-b Exploration du tokenizer et du modèle Transformer

Avant d’utiliser les pipelines simplifiés, il est important de comprendre comment un modèle Transformer reçoit le texte.

Un modèle Transformer ne comprend pas directement les phrases sous forme de texte brut.

Le texte passe d’abord par un tokenizer :

Texte brut → Tokens → Identifiants numériques → Modèle Transformer

Le tokenizer découpe le texte en mots ou sous-mots, puis transforme ces éléments en nombres appelés `input_ids`.

In [3]:
# Modèle BERT adapté au français
model_name = "dbmdz/bert-base-french-europeana-cased"

# Chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Chargement du modèle
model = AutoModel.from_pretrained(model_name)

# Exemple de phrase
texte = "Les architectures Transformers ont révolutionné le traitement du langage naturel."

# Encodage du texte
inputs = tokenizer(texte, return_tensors="pt")

print("Texte original :")
print(texte)

print("\nTokens générés :")
print(tokenizer.tokenize(texte))

print("\nIdentifiants numériques des tokens :")
print(inputs["input_ids"])

# Passage dans le modèle
outputs = model(**inputs)

print("\nForme de la dernière couche cachée :")
print(outputs.last_hidden_state.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/420 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Texte original :
Les architectures Transformers ont révolutionné le traitement du langage naturel.

Tokens générés :
['Les', 'architecture', '##s', 'Trans', '##for', '##mers', 'ont', 'révolution', '##né', 'le', 'traitement', 'du', 'langage', 'naturel', '.']

Identifiants numériques des tokens :
tensor([[    2,   519, 21475,   212,  4627,  3599, 26552,   507,  3663,  1187,
           354,  5153,   378,  7373,  6713,    18,     3]])

Forme de la dernière couche cachée :
torch.Size([1, 17, 768])


## 1-c-0 Utilisation des pipelines HuggingFace

HuggingFace propose une fonction appelée `pipeline`.

Cette fonction permet d’utiliser facilement des modèles pré-entraînés pour différentes tâches NLP, sans écrire manuellement toutes les étapes internes.

Dans cette partie, nous testons quatre tâches :

1. Analyse de sentiment
2. Classification de texte
3. Question Answering
4. Résumé automatique

## 1-c-1 Analyse de sentiment

L’analyse de sentiment consiste à déterminer l’opinion exprimée dans un texte.

Le résultat peut être positif, négatif ou parfois neutre selon le modèle utilisé.

Exemple :

Texte : "Ce projet est intéressant."  
Résultat attendu : sentiment positif

In [4]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device=device
)

textes = [
    "Ce film est absolument magnifique !",
    "Je n'ai pas du tout aimé cette expérience.",
]

for texte in textes:
    result = sentiment_pipeline(texte)
    print(f"Texte   : {texte}")
    print(f"Sentiment label : {result[0]['label']} , score : {result[0]['score']:.4f}\n")


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Texte   : Ce film est absolument magnifique !
Sentiment label : positive , score : 0.9395

Texte   : Je n'ai pas du tout aimé cette expérience.
Sentiment label : negative , score : 0.8531



### Analyse du résultat

Le modèle retourne généralement deux informations :

- `label` : la classe prédite, par exemple positif ou négatif.
- `score` : le niveau de confiance du modèle.

Plus le score est proche de 1, plus le modèle est confiant dans sa prédiction.

## 1-c-2 Classification de texte

La classification de texte consiste à attribuer une catégorie à un texte.

Dans cette expérience, nous utilisons la classification zero-shot.

Le zero-shot classification permet de classer un texte dans des catégories choisies sans entraîner un nouveau modèle.

In [5]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

texte_a_classer = "L'équipe de développement a déployé une nouvelle base de données vectorielle ce matin."

labels_possibles = [
    "informatique",
    "sport",
    "économie",
    "politique"
]

resultat_classification = classifier(
    texte_a_classer,
    candidate_labels=labels_possibles
)

print("--- Classification de Texte ---")
print("Texte à classer : ",texte_a_classer)

print("\nRésultats :")
for label, score in zip(resultat_classification["labels"], resultat_classification["scores"]):
    print(f"{label} : {score:.4f}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

--- Classification de Texte ---
Texte à classer :  L'équipe de développement a déployé une nouvelle base de données vectorielle ce matin.

Résultats :
informatique : 0.6961
politique : 0.1372
économie : 0.1362
sport : 0.0305


### Analyse du résultat

Le modèle compare le texte avec chaque label proposé.

Il retourne les labels triés selon leur probabilité.

Dans notre exemple, le label le plus logique devrait être `informatique`, car le texte parle de développement et de base de données vectorielle.

## 1-c-3 Question Answering

Le Question Answering consiste à répondre à une question à partir d’un contexte donné.

Le modèle ne répond pas librement : il cherche la réponse dans le texte fourni.

In [6]:
qa_pipeline = pipeline(
    "question-answering",
    model="deepset/xlm-roberta-base-squad2",
    device=device
)

contexte = """
L'École Nationale des Sciences Appliquées d'Al Hoceima propose une filière en Ingénierie des Données.
Au cours de la deuxième année, les étudiants réalisent un mini-projet portant sur l'exploration des modèles Transformers
et la mise en œuvre de systèmes de génération augmentée par récupération, appelés systèmes RAG.
"""

question = "Quel est le sujet du mini-projet ?"

reponse = qa_pipeline(
    question=question,
    context=contexte
)

print("Question :", question)
print("Réponse extraite :", reponse["answer"])
print("Score de confiance :", round(reponse["score"], 4))

config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Question : Quel est le sujet du mini-projet ?
Réponse extraite :  l'exploration des modèles Transformers
Score de confiance : 0.2988


### Analyse du résultat

Le modèle lit le contexte fourni, puis extrait la partie du texte qui répond le mieux à la question.

Le score indique la confiance du modèle.

Cette tâche montre que le modèle peut répondre correctement lorsqu’on lui donne un contexte pertinent.

## 1-c-4 Résumé automatique

Le résumé automatique consiste à produire une version courte d’un texte long.

L’objectif est de conserver les idées principales tout en réduisant la taille du texte.

Cette tâche est utile pour :

- Résumer des articles.
- Résumer des documents PDF.
- Extraire les informations importantes d’un corpus.

In [7]:
summarizer = pipeline(
    "summarization",
    model="lincoln/mbart-mlsum-automatic-summarization",
    device=device
)
texte_long = """
Le traitement du langage naturel a franchi un cap important avec l'introduction de l'architecture Transformer en 2017.
Contrairement aux réseaux récurrents traditionnels, qui traitent les mots de manière séquentielle,
les Transformers s'appuient sur un mécanisme d'attention. Ce mécanisme permet au modèle d'analyser les relations
entre tous les mots d'une phrase simultanément. Cette innovation a facilité l'entraînement de grands modèles de langage
sur des volumes massifs de données, ouvrant la voie aux modèles modernes utilisés dans la traduction automatique,
la génération de texte, le question answering et les systèmes RAG.
"""

resume = summarizer(
    texte_long,
    max_length=70,
    min_length=25,
    do_sample=False
)

print("Texte original :")
print(texte_long)

print("\nRésumé généré :")
print(resume[0]["summary_text"])

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/503 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

Texte original :

Le traitement du langage naturel a franchi un cap important avec l'introduction de l'architecture Transformer en 2017.
Contrairement aux réseaux récurrents traditionnels, qui traitent les mots de manière séquentielle,
les Transformers s'appuient sur un mécanisme d'attention. Ce mécanisme permet au modèle d'analyser les relations
entre tous les mots d'une phrase simultanément. Cette innovation a facilité l'entraînement de grands modèles de langage
sur des volumes massifs de données, ouvrant la voie aux modèles modernes utilisés dans la traduction automatique,
la génération de texte, le question answering et les systèmes RAG.


Résumé généré :
Le traitement du langage naturel a franchi un cap important avec l'introduction de l'architecture Transformer en 2017.


### Analyse du résultat

Le modèle de résumé automatique produit une version plus courte du texte initial.

Un bon résumé doit :

- Garder les idées principales.
- Supprimer les détails secondaires.
- Rester cohérent et compréhensible.

# Partie 2 : Systèmes RAG (Retrieval-Augmented Generation).

Dans cette partie, nous allons implémenter un système RAG simple.

L’objectif est de permettre à un modèle génératif de répondre à des questions en utilisant un corpus externe.

Le pipeline contient les étapes suivantes :

1. Importer un document PDF.
2. Extraire le texte du PDF.
3. Découper le texte en petits morceaux appelés chunks.
4. Transformer chaque chunk en embedding.
5. Indexer les embeddings avec FAISS.
6. Transformer la question utilisateur en embedding.
7. Rechercher les chunks les plus pertinents.
8. Construire un prompt augmenté.
9. Générer une réponse avec un modèle génératif.
10. Comparer les réponses avec et sans RAG.

In [8]:
# Installation des bibliothèques nécessaires
!pip install -q sentence-transformers faiss-cpu pypdf transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 32.1 MB/s eta 0:00:00


## 2-a Importation des bibliothèques

In [9]:
import numpy as np
import faiss
import torch

from pypdf import PdfReader
from google.colab import files
from sentence_transformers import SentenceTransformer
from transformers import pipeline

## 2-a-1 Importation du corpus PDF

Dans un système RAG, le corpus représente la base de connaissances externe.

Dans ce notebook, nous utilisons un fichier PDF comme corpus. Le texte du PDF sera extrait, découpé en chunks, puis transformé en embeddings.

In [10]:
print("Veuillez importer votre fichier PDF :")
uploaded = files.upload()

nom_fichier_pdf = list(uploaded.keys())[0]
print("Fichier importé :", nom_fichier_pdf)

Veuillez importer votre fichier PDF :


Saving mini projet nlp.pdf to mini projet nlp.pdf
Fichier importé : mini projet nlp.pdf


# 2-a-2 Extraction du texte

In [11]:
def extraire_texte_pdf(chemin_fichier):
    texte_complet = ""

    reader = PdfReader(chemin_fichier)

    for page in reader.pages:
        texte_page = page.extract_text()
        if texte_page:
            texte_complet += texte_page + "\n"

    return texte_complet


texte_brut = extraire_texte_pdf(nom_fichier_pdf)

print("Extraction terminée.")
print("Nombre de caractères extraits :", len(texte_brut))
print("\nAperçu du texte :")
print(texte_brut[:1000])

Extraction terminée.
Nombre de caractères extraits : 2247

Aperçu du texte :
ENSA - Al HoceimaID2
École Nationale des Sciences Appliquées- AL Hoceima
Filière : Ingénierie des Données
Deuxième Année
Mini-Projet : Transformers et Systèmes RAG
1 Partie 1 : Exploration des Transformers et tâches NLP
Objectif
Cette partie vise à familiariser les étudiants avec l’utilisation pratique des modèles Transformers
pour différentes tâches NLP, en s’appuyant sur la bibliothèque HuggingFace Transformers et un
ouvrage de référence fourni.
Travail demandé
•Exploration de la bibliothèque HuggingFace Transformers :
–Chargement de modèles pré-entraînés
–Utilisation de tokenizers
–Pipelines simplifiés pour tâches NLP
•Implémentation de plusieurs tâches NLP :
–Classification de texte
–Analyse de sentiment
–Question Answering
–Résumé automatique
1
ENSA - Al HoceimaID2
2 Partie 2 : Systèmes RAG (Retrieval-Augmented Generation)
Objectif
Comprendre, analyser et implémenter un système RAG permettant d’améliorer 

## 2-a-3 Découpage du texte en chunks

Un modèle d’embeddings ne traite pas efficacement un document entier en une seule fois.

Nous découpons donc le texte en petits morceaux appelés chunks.

Le chevauchement entre les chunks permet d’éviter de perdre une information importante lorsqu’une phrase est coupée entre deux morceaux.

In [12]:
def decouper_texte(texte, mots_par_chunk=120, chevauchement=30):
    mots = texte.split()
    chunks = []

    pas = mots_par_chunk - chevauchement

    for i in range(0, len(mots), pas):
        chunk = " ".join(mots[i:i + mots_par_chunk])

        if len(chunk.strip()) > 0:
            chunks.append(chunk)

    return chunks


documents_pdf = decouper_texte(
    texte_brut,
    mots_par_chunk=120,
    chevauchement=30
)

print("Nombre de chunks créés :", len(documents_pdf))

print("\nExemple de chunk :")
print(documents_pdf[0])

Nombre de chunks créés : 4

Exemple de chunk :
ENSA - Al HoceimaID2 École Nationale des Sciences Appliquées- AL Hoceima Filière : Ingénierie des Données Deuxième Année Mini-Projet : Transformers et Systèmes RAG 1 Partie 1 : Exploration des Transformers et tâches NLP Objectif Cette partie vise à familiariser les étudiants avec l’utilisation pratique des modèles Transformers pour différentes tâches NLP, en s’appuyant sur la bibliothèque HuggingFace Transformers et un ouvrage de référence fourni. Travail demandé •Exploration de la bibliothèque HuggingFace Transformers : –Chargement de modèles pré-entraînés –Utilisation de tokenizers –Pipelines simplifiés pour tâches NLP •Implémentation de plusieurs tâches NLP : –Classification de texte –Analyse de sentiment –Question Answering –Résumé automatique 1 ENSA - Al HoceimaID2 2 Partie 2 : Systèmes RAG (Retrieval-Augmented Generation) Objectif Comprendre, analyser et implémenter


## 2-a-4 Génération des embeddings

Un embedding est une représentation vectorielle d’un texte.

Deux textes qui ont un sens proche doivent avoir des vecteurs proches dans l’espace vectoriel.

Dans notre système RAG, chaque chunk du PDF est transformé en embedding.

In [13]:
# Chargement du modèle d’embeddings
modele_embeddings = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("Chargement du modèle d'embeddings...")
embedder = SentenceTransformer(modele_embeddings)

print("Modèle chargé :", modele_embeddings)

Chargement du modèle d'embeddings...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modèle chargé : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [14]:
# Encodage des chunks
print("Encodage des chunks en embeddings...")

document_embeddings = embedder.encode(
    documents_pdf,
    convert_to_numpy=True,
    show_progress_bar=True
)

document_embeddings = document_embeddings.astype("float32")

print("Forme de la matrice des embeddings :")
print(document_embeddings.shape)

Encodage des chunks en embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Forme de la matrice des embeddings :
(4, 384)


## 2-a-5 Indexation avec FAISS

Après avoir obtenu les embeddings, nous les ajoutons dans un index vectoriel FAISS.

FAISS permet de rechercher rapidement les chunks dont les embeddings sont les plus proches de l’embedding de la question utilisateur.

In [15]:
dimension = document_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(document_embeddings)

print("Indexation terminée.")
print("Nombre de vecteurs dans FAISS :", index.ntotal)
print("Dimension des vecteurs :", dimension)

Indexation terminée.
Nombre de vecteurs dans FAISS : 4
Dimension des vecteurs : 384


## 2-a-6 Recherche des documents pertinents

Lorsqu’un utilisateur pose une question, nous transformons cette question en embedding.

Ensuite, FAISS recherche les chunks les plus proches de cette question.

Ces chunks constituent le contexte qui sera donné au modèle génératif.

In [16]:
def rechercher_documents_pertinents(requete, k=3):
    requete_embedding = embedder.encode(
        [requete],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(requete_embedding, k)

    resultats = []

    for rang, idx in enumerate(indices[0]):
        resultats.append({
            "rang": rang + 1,
            "distance": distances[0][rang],
            "texte": documents_pdf[idx]
        })

    return resultats

Test de recherche

In [17]:
question_test = "Qu'est-ce qu'un système RAG ?"

resultats = rechercher_documents_pertinents(question_test, k=3)

print("Question :", question_test)
print("\nChunks récupérés :")

for res in resultats:
    print(f"\n--- Chunk {res['rang']} | Distance : {res['distance']:.4f} ---")
    print(res["texte"][:700])

Question : Qu'est-ce qu'un système RAG ?

Chunks récupérés :

--- Chunk 1 | Distance : 15.0685 ---
et pipeline Décrire et formaliser le pipeline complet d’un système RAG : 1. Encodage des documents sous forme d’embeddings 2. Indexation de la base de connaissances 3. Encodage de la requête utilisateur 4. Recherche des top-k documents pertinents 5. Construction du prompt augmenté 6. Génération de la réponse par le LLM 3. Implémentation Les étudiants doivent implémenter un système RAG simple comprenant : •Un corpus de documents (PDF, articles ou dataset simplifié) •Un modèle d’embeddings (ex : sentence-transformers) •Un moteur de recherche vectoriel (FAISS ou équivalent) •Un modèle génératif (API ou modèle local) 2 ENSA - Al HoceimaID2 4. Démonstration et expérimentation Le système doit être

--- Chunk 2 | Distance : 18.0499 ---
NLP : –Classification de texte –Analyse de sentiment –Question Answering –Résumé automatique 1 ENSA - Al HoceimaID2 2 Partie 2 : Systèmes RAG (Retrieval-Augmented

## 2-a-7 Génération de réponse avec un prompt augmenté

Après la récupération des chunks pertinents, nous construisons un prompt augmenté.

Le prompt contient :

1. Le contexte récupéré depuis le PDF.
2. La question de l’utilisateur.
3. Une instruction demandant au modèle de répondre uniquement à partir du contexte.

In [18]:
device = 0 if torch.cuda.is_available() else -1

print("Chargement du modèle génératif...")

llm_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    device=device
)

print("Modèle génératif chargé.")

Chargement du modèle génératif...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Modèle génératif chargé.


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## Fonction RAG complète

In [19]:
def systeme_rag_pdf(question, k=3):
    # 1. Recherche des chunks pertinents
    resultats = rechercher_documents_pertinents(question, k=k)

    # 2. Construction du contexte
    contexte = "\n\n".join([res["texte"] for res in resultats])

    # 3. Construction du prompt augmenté
    prompt = f"""
Réponds à la question en te basant uniquement sur le contexte fourni.

Contexte :
{contexte}

Question :
{question}

Réponse :
"""

    # 4. Génération de la réponse
    reponse = llm_pipeline(
        prompt,
        max_new_tokens=120,
        truncation=True
    )[0]["generated_text"]

    return reponse, resultats, prompt

## Test du système RAG

In [20]:
question = "Qu'est-ce qu'un système RAG ?"

reponse, sources, prompt_utilise = systeme_rag_pdf(question, k=3)

print("Question :", question)
print("\nRéponse générée :")
print(reponse)

print("\nSources utilisées :")
for source in sources:
    print(f"\n--- Source {source['rang']} | Distance : {source['distance']:.4f} ---")
    print(source["texte"][:500])

Question : Qu'est-ce qu'un système RAG ?

Réponse générée :
Base de connaissances 3. Answering 4. Search of top-k documents relevant 5. Construction of the prompt 6. Génération de réponse par le LLM 7. Implémentation Les étudiants doivent présenter : •Définition des systèmes RAG •Limites des LLMs sans access to external data •Comparation des approaches: –Fine-tuning –Prompt engineering –RAG •Aperçu des variants modernes des architectures RAG 2. Architecture and pipeline

Sources utilisées :

--- Source 1 | Distance : 15.0685 ---
et pipeline Décrire et formaliser le pipeline complet d’un système RAG : 1. Encodage des documents sous forme d’embeddings 2. Indexation de la base de connaissances 3. Encodage de la requête utilisateur 4. Recherche des top-k documents pertinents 5. Construction du prompt augmenté 6. Génération de la réponse par le LLM 3. Implémentation Les étudiants doivent implémenter un système RAG simple comprenant : •Un corpus de documents (PDF, articles ou dataset simplif

## 2-a-8 Comparaison avec et sans RAG

Le cahier des charges demande de comparer les réponses obtenues avec et sans RAG.

Sans RAG, le modèle répond uniquement avec ses connaissances internes.

Avec RAG, le modèle reçoit un contexte extrait du corpus PDF avant de générer la réponse.

In [21]:
def repondre_sans_rag(question):
    prompt = f"""
Réponds à la question suivante :

Question :
{question}

Réponse :
"""

    reponse = llm_pipeline(
        prompt,
        max_new_tokens=120,
        truncation=True
    )[0]["generated_text"]

    return reponse

# Comparaison finale

In [22]:
question_comparaison = "Quelle est la différence entre le fine-tuning et le RAG ?"

reponse_sans_rag = repondre_sans_rag(question_comparaison)
reponse_avec_rag, sources_rag, _ = systeme_rag_pdf(question_comparaison, k=3)

print("Question :")
print(question_comparaison)

print("\n--- Réponse SANS RAG ---")
print(reponse_sans_rag)

print("\n--- Réponse AVEC RAG ---")
print(reponse_avec_rag)

print("\n--- Sources utilisées par le RAG ---")
for source in sources_rag:
    print(f"\nSource {source['rang']} | Distance : {source['distance']:.4f}")
    print(source["texte"][:500])

Question :
Quelle est la différence entre le fine-tuning et le RAG ?

--- Réponse SANS RAG ---
What is the difference between fine-tuning and RAG ?

--- Réponse AVEC RAG ---
Objectifs : – Classification of text –Analyse de sentiment –Question Answering –Résumé automatique 1 ENSA - Al HoceimaID2 2 Part 2 : Systèmes RAG (Retrieval-Augmented Generation) Objectifs : •Analyse de sentiment –Question Answering –Résumé automatique 1 ENSA - Al HoceimaID2 2 Part 2 : Sys

--- Sources utilisées par le RAG ---

Source 1 | Distance : 11.3417
NLP : –Classification de texte –Analyse de sentiment –Question Answering –Résumé automatique 1 ENSA - Al HoceimaID2 2 Partie 2 : Systèmes RAG (Retrieval-Augmented Generation) Objectif Comprendre, analyser et implémenter un système RAG permettant d’améliorer les capacités des modèles de langage en intégrant une étape de récupération d’information externe. Travail demandé 1. État de l’art Les étudiants doivent présenter : •Définition des systèmes RAG •Limites des 